# 14 GHSL Six-City Settlement Transition Analysis

This notebook materializes proposal-facing settlement-transition outputs for the six validated metropolitan regions across all nine five-year transitions from 1975→1980 through 2015→2020.

It uses the validated 294,000-row longitudinal dataset created in Notebook 12 and runs every Spark/HDFS operation remotely through `/content/drive/MyDrive/remote_exec.sh`.

## Outputs

HDFS root:

`/data/ghsl/analytics/six_city_settlement_transitions_v1`

- `cell_transitions`: 264,600 cell-transition records
- `smod_class_matrix`: city-period SMOD transition matrix
- `settlement_group_matrix`: city-period settlement-group transition matrix
- `transition_category_summary`: proposal-facing categories
- `six_city_period_summary`: aggregate comparison across all six cities

Google Drive root:

`/content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_settlement_transitions_v1`

Run one cell at a time. All permanent HDFS writes refuse to overwrite an existing output.

In [1]:
# ============================================================
# CELL 1: DRIVE, PATHS, CONSTANTS
# ============================================================
from google.colab import drive
from pathlib import Path
import base64, csv, json, os, shutil, subprocess, time

drive.mount('/content/drive', force_remount=False)
REMOTE_EXEC=Path('/content/drive/MyDrive/remote_exec.sh')
DRIVE_SSH_KEY=Path('/content/drive/MyDrive/hadoop-hdfs.pem')
COLAB_SSH_KEY=Path('/content/hadoop-hdfs.pem')
INVENTORY_PATH=Path('/content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_inventory.json')
ARTIFACT_ROOT=Path('/content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_settlement_transitions_v1')
for p in [REMOTE_EXEC,DRIVE_SSH_KEY,INVENTORY_PATH]:
    if not p.is_file(): raise FileNotFoundError(str(p))
REMOTE_EXEC.chmod(0o700)
if COLAB_SSH_KEY.exists(): COLAB_SSH_KEY.unlink()
shutil.copy2(DRIVE_SSH_KEY,COLAB_SSH_KEY); COLAB_SSH_KEY.chmod(0o600)
ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True)
INVENTORY=json.loads(INVENTORY_PATH.read_text(encoding='utf-8'))
LONGITUDINAL='/data/ghsl/modeling/six_city_longitudinal_v1'
OUTPUT_ROOT='/data/ghsl/analytics/six_city_settlement_transitions_v1'
CELL_TRANSITIONS=OUTPUT_ROOT+'/cell_transitions'
SMOD_MATRIX=OUTPUT_ROOT+'/smod_class_matrix'
GROUP_MATRIX=OUTPUT_ROOT+'/settlement_group_matrix'
CATEGORY_SUMMARY=OUTPUT_ROOT+'/transition_category_summary'
PERIOD_SUMMARY=OUTPUT_ROOT+'/six_city_period_summary'
EXPECTED_EPOCHS=[1975,1980,1985,1990,1995,2000,2005,2010,2015,2020]
EXPECTED_TRANSITIONS=264600
EXPECTED_CITY_PERIOD_ROWS=4900
print('Artifact root:',ARTIFACT_ROOT)
print('CELL 1: SUCCESS')

Mounted at /content/drive
Artifact root: /content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_settlement_transitions_v1
CELL 1: SUCCESS


In [2]:
# ============================================================
# CELL 2: REMOTE_EXEC.SH HELPER AND CONNECTION TEST
# ============================================================
def run_remote(command,timeout=300,show_output=True,raise_on_error=True):
    r=subprocess.run([str(REMOTE_EXEC),command],text=True,capture_output=True,timeout=timeout)
    if show_output and r.stdout: print(r.stdout)
    if show_output and r.stderr: print('REMOTE STDERR:'); print(r.stderr)
    if raise_on_error and r.returncode!=0: raise RuntimeError('Remote command failed with return code '+str(r.returncode))
    return r
r=run_remote("set -e; echo REMOTE_EXEC_CONNECTION_SUCCESS; whoami; hostname",90)
if 'REMOTE_EXEC_CONNECTION_SUCCESS' not in r.stdout: raise RuntimeError('Remote connection marker missing')
print('CELL 2: SUCCESS')

REMOTE_EXEC_CONNECTION_SUCCESS
ubuntu
ip-172-31-30-94

REMOTE STDERR:

CELL 2: SUCCESS


In [3]:
# ============================================================
# CELL 3: HDFS/SPARK PREFLIGHT AND INPUT AUDIT
# ============================================================
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
for d in NameNode DataNode SecondaryNameNode; do jps | awk '{print $2}' | grep -qx "$d"; done
hdfs dfsadmin -safemode get | grep -qi 'Safe mode is OFF'
hdfs dfs -test -d /data/ghsl/modeling/six_city_longitudinal_v1
command -v spark-submit >/dev/null
python3 - <<'PY_REMOTE'
import pyspark
print('PySpark import: PASS')
PY_REMOTE
echo SETTLEMENT_TRANSITION_PREFLIGHT_SUCCESS
"""
r=run_remote(cmd,300)
if 'SETTLEMENT_TRANSITION_PREFLIGHT_SUCCESS' not in r.stdout: raise RuntimeError('Preflight failed')
if INVENTORY.get('expected_total_rows')!=294000: raise RuntimeError('Frozen inventory mismatch')
if len(INVENTORY.get('training_city_slugs',[]))!=6: raise RuntimeError('Expected six cities')
print('CELL 3: SUCCESS')

PySpark import: PASS
SETTLEMENT_TRANSITION_PREFLIGHT_SUCCESS

REMOTE STDERR:

CELL 3: SUCCESS


In [4]:
# ============================================================
# CELL 4: DISCOVER AND VALIDATE LONGITUDINAL SCHEMA
# ============================================================
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
S=/tmp/nb14_schema.py
cat > "$S" <<'PY_REMOTE'
from pyspark.sql import SparkSession,functions as F
import json
s=SparkSession.builder.appName('GHSL-Settlement-Schema-Audit').getOrCreate();s.sparkContext.setLogLevel('WARN')
try:
 d=s.read.parquet('/data/ghsl/modeling/six_city_longitudinal_v1')
 required=['city_slug','grid_cell_id','epoch','smod_class','settlement_group','is_urban','is_urban_centre','built_surface_total_m2','population_total']
 missing=[x for x in required if x not in d.columns]
 rows=d.count();keys=d.select('city_slug','epoch','grid_cell_id').distinct().count();cities=d.select('city_slug').distinct().count();epochs=sorted(r[0] for r in d.select('epoch').distinct().collect())
 if missing or rows!=294000 or keys!=294000 or cities!=6 or epochs!=[1975,1980,1985,1990,1995,2000,2005,2010,2015,2020]: raise RuntimeError(repr({'missing':missing,'rows':rows,'keys':keys,'cities':cities,'epochs':epochs}))
 classes=sorted([r[0] for r in d.select('smod_class').distinct().collect() if r[0] is not None],key=str)
 groups=sorted([r[0] for r in d.select('settlement_group').distinct().collect() if r[0] is not None],key=str)
 out={'rows':rows,'keys':keys,'cities':cities,'epochs':epochs,'smod_classes':classes,'settlement_groups':groups}
 print('SCHEMA_AUDIT_JSON_BEGIN');print(json.dumps(out,sort_keys=True));print('SCHEMA_AUDIT_JSON_END');print('SETTLEMENT_SCHEMA_AUDIT_SUCCESS')
finally:s.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 4g --conf spark.ui.enabled=false "$S"
rm -f "$S"
"""
r=run_remote(cmd,3600,show_output=False)
if 'SETTLEMENT_SCHEMA_AUDIT_SUCCESS' not in r.stdout:
    print(r.stdout[-12000:]);print(r.stderr[-6000:]);raise RuntimeError('Schema audit failed')
SCHEMA_AUDIT=json.loads(r.stdout.split('SCHEMA_AUDIT_JSON_BEGIN',1)[1].split('SCHEMA_AUDIT_JSON_END',1)[0].strip())
print(json.dumps(SCHEMA_AUDIT,indent=2))
print('CELL 4: SUCCESS')

{
  "cities": 6,
  "epochs": [
    1975,
    1980,
    1985,
    1990,
    1995,
    2000,
    2005,
    2010,
    2015,
    2020
  ],
  "keys": 294000,
  "rows": 294000,
  "settlement_groups": [
    "other_or_unknown",
    "rural",
    "urban_centre",
    "urban_cluster",
    "water"
  ],
  "smod_classes": [
    0,
    10,
    11,
    12,
    13,
    21,
    22,
    23,
    30
  ]
}
CELL 4: SUCCESS


In [8]:
# ============================================================
# CELL 5: BUILD CELL TRANSITIONS AND ALL FOUR SUMMARY DATASETS
#
# Creates:
#   - 264,600 cell-level transitions
#   - SMOD-class transition matrix
#   - Settlement-group transition matrix
#   - Transition-category summary
#   - Combined six-city period summary
#
# Safety:
#   - Refuses to overwrite the permanent output root
#   - Does not modify the source longitudinal dataset
#   - Validates all outputs before reporting success
# ============================================================

required_names = [
    "LONGITUDINAL",
    "OUTPUT_ROOT",
    "CELL_TRANSITIONS",
    "SMOD_MATRIX",
    "GROUP_MATRIX",
    "CATEGORY_SUMMARY",
    "PERIOD_SUMMARY",
    "run_remote",
]

for required_name in required_names:
    if required_name not in globals():
        raise RuntimeError(
            required_name
            + " is unavailable. Run Cells 1-4 first."
        )


payload = {
    "input": LONGITUDINAL,
    "output_root": OUTPUT_ROOT,
    "cell": CELL_TRANSITIONS,
    "smod": SMOD_MATRIX,
    "group": GROUP_MATRIX,
    "category": CATEGORY_SUMMARY,
    "period": PERIOD_SUMMARY,
}


payload_b64 = base64.b64encode(
    json.dumps(
        payload,
        sort_keys=True,
    ).encode(
        "utf-8"
    )
).decode(
    "ascii"
)


command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

PAYLOAD="/tmp/nb14_payload.json"
SCRIPT="/tmp/nb14_build_corrected.py"

echo "__PAYLOAD__" \
| base64 -d \
> "$PAYLOAD"


cat > "$SCRIPT" <<'PY_REMOTE'
from pyspark.sql import (
    SparkSession,
    functions as F,
    Window,
)

import json
import sys


payload_path = sys.argv[1]


with open(
    payload_path,
    "r",
    encoding="utf-8",
) as handle:
    payload = json.load(
        handle
    )


input_path = payload[
    "input"
]

output_root = payload[
    "output_root"
]

cell_output = payload[
    "cell"
]

smod_output = payload[
    "smod"
]

group_output = payload[
    "group"
]

category_output = payload[
    "category"
]

period_output = payload[
    "period"
]


EXPECTED_TRANSITION_ROWS = 264600
EXPECTED_UNIQUE_KEYS = 264600
EXPECTED_CITY_PERIODS = 54
EXPECTED_ROWS_PER_CITY_PERIOD = 4900
EXPECTED_ROWS_PER_PERIOD = 29400


spark = (
    SparkSession.builder
    .appName(
        "GHSL-Six-City-Settlement-Transitions"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel(
    "WARN"
)


try:
    # ========================================================
    # REFUSE TO OVERWRITE PERMANENT OUTPUTS
    # ========================================================

    filesystem = (
        spark._jvm
        .org.apache.hadoop.fs.FileSystem
        .get(
            spark._jsc.hadoopConfiguration()
        )
    )

    output_root_path = (
        spark._jvm
        .org.apache.hadoop.fs.Path(
            output_root
        )
    )

    if filesystem.exists(
        output_root_path
    ):
        raise RuntimeError(
            "Refusing to overwrite existing output root: "
            + output_root
        )


    # ========================================================
    # READ THE VALIDATED SIX-CITY LONGITUDINAL DATASET
    # ========================================================

    longitudinal = (
        spark.read
        .parquet(
            input_path
        )
    )


    required_columns = [
        "city_name",
        "city_slug",
        "grid_cell_id",
        "grid_1km_row",
        "grid_1km_col",
        "grid_min_x",
        "grid_max_x",
        "grid_min_y",
        "grid_max_y",
        "epoch",
        "smod_class",
        "settlement_group",
        "is_urban",
        "is_urban_centre",
        "built_surface_total_m2",
        "population_total",
    ]


    missing_columns = [
        column_name
        for column_name
        in required_columns
        if column_name
        not in longitudinal.columns
    ]


    if missing_columns:
        raise RuntimeError(
            "The longitudinal dataset is missing "
            "required columns: "
            + repr(
                missing_columns
            )
        )


    longitudinal_rows = (
        longitudinal.count()
    )

    if longitudinal_rows != 294000:
        raise RuntimeError(
            "Expected 294,000 longitudinal rows, found "
            + str(
                longitudinal_rows
            )
        )


    # ========================================================
    # CREATE NEXT-EPOCH VALUES
    # ========================================================

    transition_window = (
        Window
        .partitionBy(
            "city_slug",
            "grid_cell_id",
        )
        .orderBy(
            "epoch"
        )
    )


    transitions = (
        longitudinal

        .withColumn(
            "to_epoch",
            F.lead(
                "epoch"
            ).over(
                transition_window
            ),
        )

        .withColumn(
            "to_smod_class",
            F.lead(
                "smod_class"
            ).over(
                transition_window
            ),
        )

        .withColumn(
            "to_settlement_group",
            F.lead(
                "settlement_group"
            ).over(
                transition_window
            ),
        )

        .withColumn(
            "to_is_urban",
            F.lead(
                "is_urban"
            ).over(
                transition_window
            ),
        )

        .withColumn(
            "to_is_urban_centre",
            F.lead(
                "is_urban_centre"
            ).over(
                transition_window
            ),
        )

        .withColumn(
            "to_built_surface_total_m2",
            F.lead(
                F.col(
                    "built_surface_total_m2"
                ).cast(
                    "double"
                )
            ).over(
                transition_window
            ),
        )

        .withColumn(
            "to_population_total",
            F.lead(
                F.col(
                    "population_total"
                ).cast(
                    "double"
                )
            ).over(
                transition_window
            ),
        )

        .withColumnRenamed(
            "epoch",
            "from_epoch",
        )

        .withColumnRenamed(
            "smod_class",
            "from_smod_class",
        )

        .withColumnRenamed(
            "settlement_group",
            "from_settlement_group",
        )

        .withColumnRenamed(
            "is_urban",
            "from_is_urban",
        )

        .withColumnRenamed(
            "is_urban_centre",
            "from_is_urban_centre",
        )

        .withColumnRenamed(
            "built_surface_total_m2",
            "from_built_surface_total_m2",
        )

        .withColumnRenamed(
            "population_total",
            "from_population_total",
        )

        .filter(
            F.col(
                "to_epoch"
            ).isNotNull()
            & (
                (
                    F.col(
                        "to_epoch"
                    )
                    - F.col(
                        "from_epoch"
                    )
                )
                == 5
            )
        )

        .withColumn(
            "built_surface_change_m2",
            (
                F.col(
                    "to_built_surface_total_m2"
                )
                - F.col(
                    "from_built_surface_total_m2"
                ).cast(
                    "double"
                )
            ),
        )

        .withColumn(
            "population_change",
            (
                F.col(
                    "to_population_total"
                )
                - F.col(
                    "from_population_total"
                ).cast(
                    "double"
                )
            ),
        )
    )


    # ========================================================
    # NORMALIZED BOOLEAN EXPRESSIONS
    # ========================================================

    from_is_urban = (
        F.coalesce(
            F.col(
                "from_is_urban"
            ).cast(
                "boolean"
            ),
            F.lit(
                False
            ),
        )
    )

    to_is_urban = (
        F.coalesce(
            F.col(
                "to_is_urban"
            ).cast(
                "boolean"
            ),
            F.lit(
                False
            ),
        )
    )

    from_is_urban_centre = (
        F.coalesce(
            F.col(
                "from_is_urban_centre"
            ).cast(
                "boolean"
            ),
            F.lit(
                False
            ),
        )
    )

    to_is_urban_centre = (
        F.coalesce(
            F.col(
                "to_is_urban_centre"
            ).cast(
                "boolean"
            ),
            F.lit(
                False
            ),
        )
    )


    # ========================================================
    # CLASSIFY SETTLEMENT TRANSITIONS
    #
    # Order matters:
    #   1. New urban centre is more specific than rural-to-urban.
    #   2. Stable urban centre is more specific than stable urban.
    #   3. Urban intensification captures changing SMOD class
    #      while remaining urban.
    # ========================================================

    transition_category_expression = (
        F.when(
            (
                ~from_is_urban_centre
            )
            & to_is_urban_centre,
            F.lit(
                "new_urban_centre"
            ),
        )

        .when(
            (
                ~from_is_urban
            )
            & to_is_urban,
            F.lit(
                "rural_to_urban"
            ),
        )

        .when(
            from_is_urban_centre
            & to_is_urban_centre,
            F.lit(
                "stable_urban_centre"
            ),
        )

        .when(
            from_is_urban
            & to_is_urban
            & (
                F.col(
                    "to_smod_class"
                )
                != F.col(
                    "from_smod_class"
                )
            ),
            F.lit(
                "urban_intensification"
            ),
        )

        .when(
            from_is_urban
            & (
                ~to_is_urban
            ),
            F.lit(
                "urban_regression"
            ),
        )

        .when(
            (
                ~from_is_urban
            )
            & (
                ~to_is_urban
            ),
            F.lit(
                "stable_rural"
            ),
        )

        .when(
            F.col(
                "from_smod_class"
            )
            == F.col(
                "to_smod_class"
            ),
            F.lit(
                "stable_other"
            ),
        )

        .otherwise(
            F.lit(
                "other_transition"
            )
        )
    )


    transitions = (
        transitions
        .withColumn(
            "transition_category",
            transition_category_expression,
        )
    )


    # ========================================================
    # SELECT FINAL CELL-TRANSITION SCHEMA
    # ========================================================

    transition_columns = [
        "city_name",
        "city_slug",
        "grid_cell_id",
        "grid_1km_row",
        "grid_1km_col",
        "grid_min_x",
        "grid_max_x",
        "grid_min_y",
        "grid_max_y",
        "from_epoch",
        "to_epoch",
        "from_smod_class",
        "to_smod_class",
        "from_settlement_group",
        "to_settlement_group",
        "from_is_urban",
        "to_is_urban",
        "from_is_urban_centre",
        "to_is_urban_centre",
        "from_built_surface_total_m2",
        "to_built_surface_total_m2",
        "built_surface_change_m2",
        "from_population_total",
        "to_population_total",
        "population_change",
        "transition_category",
    ]


    transitions = (
        transitions.select(
            *transition_columns
        )
    )


    # ========================================================
    # VALIDATE CELL-LEVEL TRANSITIONS
    # ========================================================

    transition_rows = (
        transitions.count()
    )

    unique_transition_keys = (
        transitions
        .select(
            "city_slug",
            "grid_cell_id",
            "from_epoch",
            "to_epoch",
        )
        .distinct()
        .count()
    )

    city_period_counts = (
        transitions
        .groupBy(
            "city_slug",
            "from_epoch",
            "to_epoch",
        )
        .count()
    )

    city_period_count = (
        city_period_counts.count()
    )

    invalid_city_periods = (
        city_period_counts
        .filter(
            F.col(
                "count"
            )
            != EXPECTED_ROWS_PER_CITY_PERIOD
        )
        .count()
    )

    null_state_count = (
        transitions
        .filter(
            F.col(
                "from_smod_class"
            ).isNull()
            | F.col(
                "to_smod_class"
            ).isNull()
            | F.col(
                "from_settlement_group"
            ).isNull()
            | F.col(
                "to_settlement_group"
            ).isNull()
        )
        .count()
    )


    if (
        transition_rows
        != EXPECTED_TRANSITION_ROWS
    ):
        raise RuntimeError(
            "Expected 264,600 cell transitions, found "
            + str(
                transition_rows
            )
        )


    if (
        unique_transition_keys
        != EXPECTED_UNIQUE_KEYS
    ):
        raise RuntimeError(
            "Expected 264,600 unique transition keys, found "
            + str(
                unique_transition_keys
            )
        )


    if (
        city_period_count
        != EXPECTED_CITY_PERIODS
    ):
        raise RuntimeError(
            "Expected 54 city-period groups, found "
            + str(
                city_period_count
            )
        )


    if invalid_city_periods != 0:
        raise RuntimeError(
            "One or more city-period groups do not contain "
            "exactly 4,900 cells."
        )


    if null_state_count != 0:
        raise RuntimeError(
            "Found "
            + str(
                null_state_count
            )
            + " transitions with null settlement states."
        )


    # ========================================================
    # AGGREGATION HELPER
    # ========================================================

    def build_matrix(
        grouping_columns,
        denominator,
    ):
        return (
            transitions
            .groupBy(
                *grouping_columns
            )
            .agg(
                F.count(
                    "*"
                ).alias(
                    "cell_count"
                ),

                F.sum(
                    "built_surface_change_m2"
                ).alias(
                    "built_surface_change_m2"
                ),

                F.avg(
                    "built_surface_change_m2"
                ).alias(
                    "mean_built_surface_change_m2"
                ),

                F.sum(
                    "population_change"
                ).alias(
                    "population_change"
                ),

                F.avg(
                    "population_change"
                ).alias(
                    "mean_population_change"
                ),
            )

            .withColumn(
                "cell_share",
                (
                    F.col(
                        "cell_count"
                    )
                    / F.lit(
                        float(
                            denominator
                        )
                    )
                ),
            )
        )


    # ========================================================
    # BUILD CITY-PERIOD MATRICES
    # ========================================================

    smod_matrix = build_matrix(
        [
            "city_slug",
            "from_epoch",
            "to_epoch",
            "from_smod_class",
            "to_smod_class",
        ],
        EXPECTED_ROWS_PER_CITY_PERIOD,
    )


    settlement_group_matrix = build_matrix(
        [
            "city_slug",
            "from_epoch",
            "to_epoch",
            "from_settlement_group",
            "to_settlement_group",
        ],
        EXPECTED_ROWS_PER_CITY_PERIOD,
    )


    transition_category_summary = build_matrix(
        [
            "city_slug",
            "from_epoch",
            "to_epoch",
            "transition_category",
        ],
        EXPECTED_ROWS_PER_CITY_PERIOD,
    )


    # ========================================================
    # BUILD COMBINED SIX-CITY PERIOD SUMMARY
    # ========================================================

    six_city_period_summary = (
        transitions
        .groupBy(
            "from_epoch",
            "to_epoch",
            "transition_category",
        )
        .agg(
            F.count(
                "*"
            ).alias(
                "cell_count"
            ),

            F.sum(
                "built_surface_change_m2"
            ).alias(
                "built_surface_change_m2"
            ),

            F.avg(
                "built_surface_change_m2"
            ).alias(
                "mean_built_surface_change_m2"
            ),

            F.sum(
                "population_change"
            ).alias(
                "population_change"
            ),

            F.avg(
                "population_change"
            ).alias(
                "mean_population_change"
            ),
        )

        .withColumn(
            "cell_share",
            (
                F.col(
                    "cell_count"
                )
                / F.lit(
                    float(
                        EXPECTED_ROWS_PER_PERIOD
                    )
                )
            ),
        )
    )


    # ========================================================
    # VALIDATE SUMMARY TOTALS BEFORE WRITING
    # ========================================================

    category_totals = (
        transition_category_summary
        .groupBy(
            "city_slug",
            "from_epoch",
            "to_epoch",
        )
        .agg(
            F.sum(
                "cell_count"
            ).alias(
                "total_cells"
            ),

            F.sum(
                "cell_share"
            ).alias(
                "total_share"
            ),
        )
    )


    invalid_category_totals = (
        category_totals
        .filter(
            (
                F.col(
                    "total_cells"
                )
                != EXPECTED_ROWS_PER_CITY_PERIOD
            )
            | (
                F.abs(
                    F.col(
                        "total_share"
                    )
                    - F.lit(
                        1.0
                    )
                )
                > F.lit(
                    0.000000001
                )
            )
        )
        .count()
    )


    period_totals = (
        six_city_period_summary
        .groupBy(
            "from_epoch",
            "to_epoch",
        )
        .agg(
            F.sum(
                "cell_count"
            ).alias(
                "total_cells"
            ),

            F.sum(
                "cell_share"
            ).alias(
                "total_share"
            ),
        )
    )


    invalid_period_totals = (
        period_totals
        .filter(
            (
                F.col(
                    "total_cells"
                )
                != EXPECTED_ROWS_PER_PERIOD
            )
            | (
                F.abs(
                    F.col(
                        "total_share"
                    )
                    - F.lit(
                        1.0
                    )
                )
                > F.lit(
                    0.000000001
                )
            )
        )
        .count()
    )


    if invalid_category_totals != 0:
        raise RuntimeError(
            "One or more city-period category summaries "
            "do not sum to 4,900 cells and a share of 1."
        )


    if invalid_period_totals != 0:
        raise RuntimeError(
            "One or more six-city period summaries "
            "do not sum to 29,400 cells and a share of 1."
        )


    # ========================================================
    # WRITE PERMANENT HDFS OUTPUTS
    # ========================================================

    (
        transitions
        .repartition(
            "city_slug",
            "to_epoch",
        )
        .write
        .mode(
            "errorifexists"
        )
        .partitionBy(
            "city_slug",
            "to_epoch",
        )
        .parquet(
            cell_output
        )
    )


    (
        smod_matrix
        .write
        .mode(
            "errorifexists"
        )
        .parquet(
            smod_output
        )
    )


    (
        settlement_group_matrix
        .write
        .mode(
            "errorifexists"
        )
        .parquet(
            group_output
        )
    )


    (
        transition_category_summary
        .write
        .mode(
            "errorifexists"
        )
        .parquet(
            category_output
        )
    )


    (
        six_city_period_summary
        .write
        .mode(
            "errorifexists"
        )
        .parquet(
            period_output
        )
    )


    # ========================================================
    # POST-WRITE REREAD VALIDATION
    # ========================================================

    written_transition_rows = (
        spark.read
        .parquet(
            cell_output
        )
        .count()
    )

    written_transition_keys = (
        spark.read
        .parquet(
            cell_output
        )
        .select(
            "city_slug",
            "grid_cell_id",
            "from_epoch",
            "to_epoch",
        )
        .distinct()
        .count()
    )


    if (
        written_transition_rows
        != EXPECTED_TRANSITION_ROWS
    ):
        raise RuntimeError(
            "Post-write transition row validation failed."
        )


    if (
        written_transition_keys
        != EXPECTED_UNIQUE_KEYS
    ):
        raise RuntimeError(
            "Post-write transition-key validation failed."
        )


    # ========================================================
    # FINAL BUILD SUMMARY
    # ========================================================

    categories = sorted(
        row[
            0
        ]
        for row
        in transition_category_summary
        .select(
            "transition_category"
        )
        .distinct()
        .collect()
    )


    output = {
        "cell_transition_rows": (
            written_transition_rows
        ),
        "unique_keys": (
            written_transition_keys
        ),
        "city_periods": (
            city_period_count
        ),
        "null_settlement_states": (
            null_state_count
        ),
        "smod_matrix_rows": (
            smod_matrix.count()
        ),
        "settlement_group_matrix_rows": (
            settlement_group_matrix.count()
        ),
        "category_summary_rows": (
            transition_category_summary.count()
        ),
        "period_summary_rows": (
            six_city_period_summary.count()
        ),
        "categories": categories,
        "output_root": (
            output_root
        ),
    }


    print(
        "TRANSITION_BUILD_JSON_BEGIN"
    )

    print(
        json.dumps(
            output,
            sort_keys=True,
        )
    )

    print(
        "TRANSITION_BUILD_JSON_END"
    )

    print(
        "SIX_CITY_SETTLEMENT_TRANSITION_BUILD_SUCCESS"
    )


finally:
    spark.stop()
PY_REMOTE


spark-submit \
    --master local[2] \
    --driver-memory 5g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=24 \
    --conf spark.driver.maxResultSize=1g \
    "$SCRIPT" \
    "$PAYLOAD"


rm -f \
    "$SCRIPT" \
    "$PAYLOAD"

echo
echo "Temporary build files removed: PASS"
echo "Source HDFS dataset modified: NO"
echo "CORRECTED_CELL_5_REMOTE_SUCCESS"
'''


command = (
    command.replace(
        "__PAYLOAD__",
        payload_b64,
    )
)


# ============================================================
# RUN WITH VISIBLE OUTPUT
# ============================================================

build_result = run_remote(
    command,
    timeout=7200,
    show_output=True,
)


if (
    "SIX_CITY_SETTLEMENT_TRANSITION_BUILD_SUCCESS"
    not in build_result.stdout
):
    raise RuntimeError(
        "The corrected transition build did not "
        "produce its Spark success marker."
    )


if (
    "CORRECTED_CELL_5_REMOTE_SUCCESS"
    not in build_result.stdout
):
    raise RuntimeError(
        "The corrected transition build did not "
        "produce its remote completion marker."
    )


# ============================================================
# PARSE BUILD SUMMARY
# ============================================================

begin_marker = (
    "TRANSITION_BUILD_JSON_BEGIN"
)

end_marker = (
    "TRANSITION_BUILD_JSON_END"
)


begin_index = (
    build_result.stdout.find(
        begin_marker
    )
)

end_index = (
    build_result.stdout.find(
        end_marker
    )
)


if begin_index < 0:
    raise RuntimeError(
        "Transition-build JSON begin marker was not found."
    )


if end_index < 0:
    raise RuntimeError(
        "Transition-build JSON end marker was not found."
    )


if end_index <= begin_index:
    raise RuntimeError(
        "Transition-build JSON markers are in the wrong order."
    )


json_start = (
    begin_index
    + len(
        begin_marker
    )
)


build_json = (
    build_result.stdout[
        json_start:
        end_index
    ]
    .strip()
)


BUILD_SUMMARY = json.loads(
    build_json
)


# ============================================================
# LOCAL VALIDATION
# ============================================================

if (
    BUILD_SUMMARY.get(
        "cell_transition_rows"
    )
    != 264600
):
    raise RuntimeError(
        "Expected 264,600 written transitions, found "
        + str(
            BUILD_SUMMARY.get(
                "cell_transition_rows"
            )
        )
    )


if (
    BUILD_SUMMARY.get(
        "unique_keys"
    )
    != 264600
):
    raise RuntimeError(
        "Expected 264,600 unique transition keys."
    )


if (
    BUILD_SUMMARY.get(
        "city_periods"
    )
    != 54
):
    raise RuntimeError(
        "Expected 54 city-period groups."
    )


if (
    BUILD_SUMMARY.get(
        "null_settlement_states"
    )
    != 0
):
    raise RuntimeError(
        "Null settlement states remain in the output."
    )


# ============================================================
# FINAL REPORT
# ============================================================

print(
    json.dumps(
        BUILD_SUMMARY,
        indent=2,
        sort_keys=True,
    )
)

print(
    "Cell transitions:",
    BUILD_SUMMARY[
        "cell_transition_rows"
    ],
)

print(
    "Unique transition keys:",
    BUILD_SUMMARY[
        "unique_keys"
    ],
)

print(
    "City-period groups:",
    BUILD_SUMMARY[
        "city_periods"
    ],
)

print(
    "Transition categories:",
    BUILD_SUMMARY[
        "categories"
    ],
)

print(
    "Source HDFS dataset modified: NO"
)

print(
    "CELL 5: SUCCESS"
)

TRANSITION_BUILD_JSON_BEGIN
{"categories": ["new_urban_centre", "rural_to_urban", "stable_other", "stable_rural", "stable_urban_centre", "urban_intensification", "urban_regression"], "category_summary_rows": 364, "cell_transition_rows": 264600, "city_periods": 54, "null_settlement_states": 0, "output_root": "/data/ghsl/analytics/six_city_settlement_transitions_v1", "period_summary_rows": 63, "settlement_group_matrix_rows": 466, "smod_matrix_rows": 1371, "unique_keys": 264600}
TRANSITION_BUILD_JSON_END
SIX_CITY_SETTLEMENT_TRANSITION_BUILD_SUCCESS

Temporary build files removed: PASS
Source HDFS dataset modified: NO
CORRECTED_CELL_5_REMOTE_SUCCESS

REMOTE STDERR:
26/11/15 06:32:40 INFO SparkContext: Running Spark version 3.5.9
26/11/15 06:32:40 INFO SparkContext: OS info Linux, 7.0.0-1016-aws, amd64
26/11/15 06:32:40 INFO SparkContext: Java version 11.0.32.1
26/11/15 06:32:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where

Obtained seven observed categories:

new_urban_centre
rural_to_urban
stable_other
stable_rural
stable_urban_centre
urban_intensification
urban_regression

other_transition did not appear because none of the 264,600 records required that fallback category. That is fine.

Permanent HDFS results
Plain Text
/data/ghsl/analytics/six_city_settlement_transitions_v1/cell_transitions
/data/ghsl/analytics/six_city_settlement_transitions_v1/smod_class_matrix
/data/ghsl/analytics/six_city_settlement_transitions_v1/settlement_group_matrix
/data/ghsl/analytics/six_city_settlement_transitions_v1/transition_category_summary
/data/ghsl/analytics/six_city_settlement_transitions_v1/six_city_period_summary

In [6]:
# ============================================================
# CELL 5 FAILURE DIAGNOSTIC
#
# Read-only:
#   - Does not rerun Spark
#   - Does not delete partial outputs
#   - Does not modify source datasets
# ============================================================

diagnostic_command = r'''
source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

OUTPUT_ROOT="/data/ghsl/analytics/six_city_settlement_transitions_v1"

echo "============================================================"
echo "CELL 5 SETTLEMENT-TRANSITION FAILURE DIAGNOSTIC"
echo "============================================================"

echo
echo "========== PRESERVED TEMPORARY FILES =========="

for file_path in \
    /tmp/nb14_payload.json \
    /tmp/nb14_build.py
do
    if [ -f "$file_path" ]; then
        echo "FOUND: $file_path"
        ls -lh "$file_path"
    else
        echo "MISSING: $file_path"
    fi
done

echo
echo "========== OUTPUT ROOT =========="

if hdfs dfs -test -e "$OUTPUT_ROOT"; then
    echo "OUTPUT_ROOT_EXISTS: YES"

    echo
    echo "Top-level output inventory:"
    hdfs dfs -ls "$OUTPUT_ROOT" || true

    echo
    echo "Recursive output inventory:"
    hdfs dfs -ls -R "$OUTPUT_ROOT" \
        | head -300 \
        || true
else
    echo "OUTPUT_ROOT_EXISTS: NO"
fi

echo
echo "========== EXPECTED OUTPUT DATASETS =========="

for dataset_name in \
    cell_transitions \
    smod_class_matrix \
    settlement_group_matrix \
    transition_category_summary \
    six_city_period_summary
do
    dataset_path="$OUTPUT_ROOT/$dataset_name"

    echo
    echo "Dataset: $dataset_name"

    if hdfs dfs -test -e "$dataset_path"; then
        echo "STATUS: PRESENT"

        hdfs dfs -count -h \
            "$dataset_path" \
            || true

        if hdfs dfs -test -e \
            "$dataset_path/_SUCCESS"
        then
            echo "_SUCCESS: PRESENT"
        else
            echo "_SUCCESS: MISSING"
        fi
    else
        echo "STATUS: MISSING"
    fi
done

echo
echo "========== HDFS HEALTH =========="

hdfs dfsadmin -safemode get

hdfs dfsadmin -report \
    | sed -n '1,35p'

echo
echo "========== LOCAL DISK =========="

df -h / /tmp

echo
echo "========== RECENT MEMORY EVENTS =========="

dmesg 2>/dev/null \
    | grep -Ei \
        "out of memory|oom|killed process" \
    | tail -20 \
    || true

echo
echo "CELL_5_DIAGNOSTIC_COMPLETE"
'''

diagnostic_result = run_remote(
    diagnostic_command,
    timeout=600,
    show_output=True,
)

print(
    "Return code:",
    diagnostic_result.returncode,
)

print(
    "CELL 5 DIAGNOSTIC: SUCCESS"
)

CELL 5 SETTLEMENT-TRANSITION FAILURE DIAGNOSTIC

========== PRESERVED TEMPORARY FILES ==========
FOUND: /tmp/nb14_payload.json
-rw-rw-r-- 1 ubuntu ubuntu 584 Nov 15 06:29 /tmp/nb14_payload.json
FOUND: /tmp/nb14_build.py
-rw-rw-r-- 1 ubuntu ubuntu 5.8K Nov 15 06:29 /tmp/nb14_build.py

========== OUTPUT ROOT ==========
OUTPUT_ROOT_EXISTS: NO

========== EXPECTED OUTPUT DATASETS ==========

Dataset: cell_transitions
STATUS: MISSING

Dataset: smod_class_matrix
STATUS: MISSING

Dataset: settlement_group_matrix
STATUS: MISSING

Dataset: transition_category_summary
STATUS: MISSING

Dataset: six_city_period_summary
STATUS: MISSING

========== HDFS HEALTH ==========
Safe mode is OFF
Configured Capacity: 82085269504 (76.45 GB)
Present Capacity: 63971082240 (59.58 GB)
DFS Remaining: 56688652288 (52.80 GB)
DFS Used: 7282429952 (6.78 GB)
DFS Used%: 11.38%
Replicated Blocks:
	Under replicated blocks: 1
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0

In [7]:
# ============================================================
# CELL 5A: VISIBLE RETRY OF PRESERVED TRANSITION BUILD
#
# Run only when:
#   OUTPUT_ROOT_EXISTS: NO
#
# It reuses the exact preserved script and payload from Cell 5.
# ============================================================

visible_retry_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

SCRIPT="/tmp/nb14_build.py"
PAYLOAD="/tmp/nb14_payload.json"
OUTPUT_ROOT="/data/ghsl/analytics/six_city_settlement_transitions_v1"

echo "============================================================"
echo "VISIBLE CELL 5 SETTLEMENT-TRANSITION RETRY"
echo "============================================================"

if [ ! -f "$SCRIPT" ]; then
    echo "ERROR: Preserved build script is missing."
    exit 20
fi

if [ ! -f "$PAYLOAD" ]; then
    echo "ERROR: Preserved payload is missing."
    exit 21
fi

if hdfs dfs -test -e "$OUTPUT_ROOT"; then
    echo "ERROR: Permanent output root already exists:"
    echo "$OUTPUT_ROOT"
    echo "Nothing was deleted."
    exit 22
fi

echo
echo "Build script:"
ls -lh "$SCRIPT"

echo
echo "Payload:"
cat "$PAYLOAD"

echo
echo "========== STARTING VISIBLE SPARK RETRY =========="

spark-submit \
    --master local[2] \
    --driver-memory 5g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=24 \
    --conf spark.driver.maxResultSize=1g \
    "$SCRIPT" \
    "$PAYLOAD"
'''

visible_retry_result = subprocess.run(
    [
        str(REMOTE_EXEC),
        visible_retry_command,
    ],
    text=True,
    capture_output=True,
    timeout=7200,
)

print(
    "REMOTE RETURN CODE:",
    visible_retry_result.returncode,
)

print()
print(
    "================ REMOTE STDOUT ================"
)

print(
    visible_retry_result.stdout
)

print()
print(
    "================ REMOTE STDERR ================"
)

print(
    visible_retry_result.stderr
)

if visible_retry_result.returncode == 0:
    print(
        "VISIBLE CELL 5 RETRY: SUCCESS"
    )
else:
    print(
        "VISIBLE CELL 5 RETRY: FAILED"
    )

REMOTE RETURN CODE: 1

================ REMOTE STDOUT ================
VISIBLE CELL 5 SETTLEMENT-TRANSITION RETRY

Build script:
-rw-rw-r-- 1 ubuntu ubuntu 5.8K Nov 15 06:29 /tmp/nb14_build.py

Payload:
{"input": "/data/ghsl/modeling/six_city_longitudinal_v1", "output_root": "/data/ghsl/analytics/six_city_settlement_transitions_v1", "cell": "/data/ghsl/analytics/six_city_settlement_transitions_v1/cell_transitions", "smod": "/data/ghsl/analytics/six_city_settlement_transitions_v1/smod_class_matrix", "group": "/data/ghsl/analytics/six_city_settlement_transitions_v1/settlement_group_matrix", "category": "/data/ghsl/analytics/six_city_settlement_transitions_v1/transition_category_summary", "period": "/data/ghsl/analytics/six_city_settlement_transitions_v1/six_city_period_summary"}
========== STARTING VISIBLE SPARK RETRY ==========
  File "/tmp/nb14_build.py", line 27
    .when((~F.coalesce(F.col('from_is_urban_centre').cast('boolean'),F.lit(False))) & F.coalesce(F.col('to_is_urban_centre')

In [9]:
# ============================================================
# CELL 6: PROPOSAL-FACING QUALITY GATES
# ============================================================
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
S=/tmp/nb14_validate.py
cat > "$S" <<'PY_REMOTE'
from pyspark.sql import SparkSession,functions as F
import json
s=SparkSession.builder.appName('GHSL-Settlement-Transition-Validation').getOrCreate();s.sparkContext.setLogLevel('WARN')
try:
 root='/data/ghsl/analytics/six_city_settlement_transitions_v1'
 c=s.read.parquet(root+'/cell_transitions');sm=s.read.parquet(root+'/smod_class_matrix');gm=s.read.parquet(root+'/settlement_group_matrix');cs=s.read.parquet(root+'/transition_category_summary');ps=s.read.parquet(root+'/six_city_period_summary')
 nulls=c.filter(F.col('from_smod_class').isNull()|F.col('to_smod_class').isNull()|F.col('from_settlement_group').isNull()|F.col('to_settlement_group').isNull()).count()
 totals=[r.asDict() for r in cs.groupBy('city_slug','from_epoch','to_epoch').agg(F.sum('cell_count').alias('cells'),F.sum('cell_share').alias('share')).collect()]
 bad=[x for x in totals if x['cells']!=4900 or abs(x['share']-1.0)>1e-9]
 period=[r.asDict() for r in ps.groupBy('from_epoch','to_epoch').agg(F.sum('cell_count').alias('cells'),F.sum('cell_share').alias('share')).collect()]
 badp=[x for x in period if x['cells']!=29400 or abs(x['share']-1.0)>1e-9]
 if nulls or bad or badp or len(totals)!=54 or len(period)!=9: raise RuntimeError(repr({'null_transition_classes':nulls,'bad_city_periods':bad,'bad_periods':badp,'city_periods':len(totals),'periods':len(period)}))
 out={'cell_rows':c.count(),'smod_rows':sm.count(),'group_rows':gm.count(),'category_rows':cs.count(),'period_rows':ps.count(),'null_transition_classes':nulls,'validated_city_periods':len(totals),'validated_periods':len(period)}
 print('QUALITY_JSON_BEGIN');print(json.dumps(out,sort_keys=True));print('QUALITY_JSON_END');print('SETTLEMENT_TRANSITION_QUALITY_GATES_SUCCESS')
finally:s.stop()
PY_REMOTE
spark-submit --master local[2] --driver-memory 4g --conf spark.ui.enabled=false "$S"
rm -f "$S"
"""
r=run_remote(cmd,3600,show_output=False)
if 'SETTLEMENT_TRANSITION_QUALITY_GATES_SUCCESS' not in r.stdout:
    print(r.stdout[-12000:]);print(r.stderr[-6000:]);raise RuntimeError('Quality gates failed')
QUALITY=json.loads(r.stdout.split('QUALITY_JSON_BEGIN',1)[1].split('QUALITY_JSON_END',1)[0].strip())
print(json.dumps(QUALITY,indent=2))
print('CELL 6: SUCCESS')

{
  "category_rows": 364,
  "cell_rows": 264600,
  "group_rows": 466,
  "null_transition_classes": 0,
  "period_rows": 63,
  "smod_rows": 1371,
  "validated_city_periods": 54,
  "validated_periods": 9
}
CELL 6: SUCCESS


In [10]:
# ============================================================
# CELL 7: EXPORT FOUR SUMMARY TABLES TO GOOGLE DRIVE
# Binary geometry is not part of these summary tables.
# ============================================================
exports=[('smod_class_matrix',SMOD_MATRIX),('settlement_group_matrix',GROUP_MATRIX),('transition_category_summary',CATEGORY_SUMMARY),('six_city_period_summary',PERIOD_SUMMARY)]
REMOTE_TMP='/tmp/nb14_transition_exports'
# Spark's absolute /tmp uses HDFS here; make exports and then cat through HDFS.
payload_b64=base64.b64encode(json.dumps({'exports':exports,'root':REMOTE_TMP}).encode()).decode()
cmd=r"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"
hdfs dfs -rm -r -skipTrash /tmp/nb14_transition_exports >/dev/null 2>&1 || true
P=/tmp/nb14_export_payload.json;S=/tmp/nb14_export.py
echo __P__ | base64 -d > "$P"
cat > "$S" <<'PY_REMOTE'
from pyspark.sql import SparkSession
import json,sys
p=json.load(open(sys.argv[1]));s=SparkSession.builder.appName('GHSL-Settlement-Transition-Exports').getOrCreate();s.sparkContext.setLogLevel('WARN')
try:
 for name,path in p['exports']:
  d=s.read.parquet(path);d.orderBy(*[x for x in ['from_epoch','to_epoch','city_slug','transition_category','from_smod_class','to_smod_class','from_settlement_group','to_settlement_group'] if x in d.columns]).coalesce(1).write.mode('overwrite').option('header',True).csv(p['root']+'/'+name)
 print('TRANSITION_EXPORT_SPARK_SUCCESS')
finally:s.stop()
PY_REMOTE
spark-submit --master local[1] --driver-memory 3g --conf spark.ui.enabled=false "$S" "$P"
rm -f "$S" "$P"
""".replace('__P__',payload_b64)
r=run_remote(cmd,3600)
if 'TRANSITION_EXPORT_SPARK_SUCCESS' not in r.stdout: raise RuntimeError('Export job failed')
export_manifest=[]
for name,_ in exports:
    read_cmd="source /home/ubuntu/ghsl-sedona-env.sh; export HADOOP_HOME=/usr/local/hadoop; export PATH=$HADOOP_HOME/bin:$PATH; P=$(hdfs dfs -find '"+REMOTE_TMP+'/'+name+"' -name 'part-*.csv' | head -1); test -n \"$P\"; hdfs dfs -cat \"$P\""
    x=run_remote(read_cmd,1200,show_output=False)
    path=ARTIFACT_ROOT/(name+'.csv');path.write_text(x.stdout,encoding='utf-8')
    lines=sum(1 for _ in path.open(encoding='utf-8'))
    if lines<2: raise RuntimeError('Empty export '+name)
    export_manifest.append({'name':name,'path':str(path),'data_rows':lines-1,'size_bytes':path.stat().st_size})
    print('Saved:',path,'rows=',lines-1)
(ARTIFACT_ROOT/'export_manifest.json').write_text(json.dumps(export_manifest,indent=2,sort_keys=True)+'\n')
run_remote("source /home/ubuntu/ghsl-sedona-env.sh; export HADOOP_HOME=/usr/local/hadoop; export PATH=$HADOOP_HOME/bin:$PATH; hdfs dfs -rm -r -skipTrash /tmp/nb14_transition_exports; echo TEMP_EXPORT_CLEANUP_SUCCESS",600)
print('CELL 7: SUCCESS')

TRANSITION_EXPORT_SPARK_SUCCESS

REMOTE STDERR:
26/11/15 06:36:04 INFO SparkContext: Running Spark version 3.5.9
26/11/15 06:36:04 INFO SparkContext: OS info Linux, 7.0.0-1016-aws, amd64
26/11/15 06:36:04 INFO SparkContext: Java version 11.0.32.1
26/11/15 06:36:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/15 06:36:05 INFO ResourceUtils: ==============================================================
26/11/15 06:36:05 INFO ResourceUtils: No custom resources configured for spark.driver.
26/11/15 06:36:05 INFO ResourceUtils: ==============================================================
26/11/15 06:36:05 INFO SparkContext: Submitted application: GHSL-Settlement-Transition-Exports
26/11/15 06:36:05 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: o

Saved: /content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_settlement_transitions_v1/smod_class_matrix.csv rows= 1371
Saved: /content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_settlement_transitions_v1/settlement_group_matrix.csv rows= 466
Saved: /content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_settlement_transitions_v1/transition_category_summary.csv rows= 364
Saved: /content/drive/MyDrive/GHSL_Urban_Growth_Model/six_city_settlement_transitions_v1/six_city_period_summary.csv rows= 63

In [11]:
# ============================================================
# CELL 8: DISPLAY KEY SIX-CITY TRANSITION RESULTS
# ============================================================
import pandas as pd
category_df=pd.read_csv(ARTIFACT_ROOT/'transition_category_summary.csv')
period_df=pd.read_csv(ARTIFACT_ROOT/'six_city_period_summary.csv')
print('SIX-CITY PERIOD SUMMARY')
display(period_df.sort_values(['to_epoch','cell_count'],ascending=[True,False]).reset_index(drop=True).style.format({'cell_share':'{:.2%}','built_surface_change_m2':'{:,.0f}','mean_built_surface_change_m2':'{:,.2f}','population_change':'{:,.0f}','mean_population_change':'{:,.2f}'}))
print('RURAL-TO-URBAN BY CITY AND PERIOD')
r2u=category_df[category_df.transition_category=='rural_to_urban'].sort_values(['to_epoch','cell_count'],ascending=[True,False]).reset_index(drop=True)
display(r2u.style.format({'cell_share':'{:.2%}','built_surface_change_m2':'{:,.0f}','mean_built_surface_change_m2':'{:,.2f}','population_change':'{:,.0f}','mean_population_change':'{:,.2f}'}))
print('CELL 8: SUCCESS')

SIX-CITY PERIOD SUMMARY


,from_epoch,to_epoch,transition_category,cell_count,built_surface_change_m2,mean_built_surface_change_m2,population_change,mean_population_change,cell_share
0,1975,1980,stable_rural,24089,"15,563,549",646.09,"330,848",13.73,81.94%
1,1975,1980,stable_urban_centre,2783,"31,830,283","11,437.40","4,481,175","1,610.20",9.47%
2,1975,1980,stable_other,1325,"8,650,745","6,528.86","245,473",185.26,4.51%
3,1975,1980,rural_to_urban,799,"3,965,944","4,963.63","247,551",309.83,2.72%
4,1975,1980,new_urban_centre,273,"2,544,342","9,319.93","179,848",658.78,0.93%
5,1975,1980,urban_intensification,83,"830,887","10,010.69","44,298",533.71,0.28%
6,1975,1980,urban_regression,48,"65,678","1,368.29","-2,826",-58.87,0.16%
7,1980,1985,stable_rural,23293,"19,506,644",837.45,"327,532",14.06,79.23%
8,1980,1985,stable_urban_centre,3051,"41,885,827","13,728.56","5,125,323","1,679.88",10.38%
9,1980,1985,stable_other,1723,"13,170,452","7,643.91","388,023",225.20,5.86%


RURAL-TO-URBAN BY CITY AND PERIOD


,city_slug,from_epoch,to_epoch,transition_category,cell_count,built_surface_change_m2,mean_built_surface_change_m2,population_change,mean_population_change,cell_share
0,pune_metro,1975,1980,rural_to_urban,190,"978,974","5,152.49","70,904",373.18,3.88%
1,delhi_ncr,1975,1980,rural_to_urban,144,"1,060,942","7,367.65","27,751",192.71,2.94%
2,bengaluru,1975,1980,rural_to_urban,142,"555,246","3,910.18","40,374",284.32,2.90%
3,chennai_metro,1975,1980,rural_to_urban,128,"414,518","3,238.42","57,386",448.33,2.61%
4,hyderabad,1975,1980,rural_to_urban,105,"487,572","4,643.54","24,221",230.68,2.14%
5,mumbai_metro,1975,1980,rural_to_urban,90,"468,692","5,207.69","26,915",299.06,1.84%
6,bengaluru,1980,1985,rural_to_urban,176,"663,015","3,767.13","27,003",153.43,3.59%
7,delhi_ncr,1980,1985,rural_to_urban,149,"1,230,770","8,260.20","30,683",205.93,3.04%
8,pune_metro,1980,1985,rural_to_urban,133,"822,961","6,187.68","27,785",208.91,2.71%
9,chennai_metro,1980,1985,rural_to_urban,126,"350,665","2,783.06","18,797",149.18,2.57%


CELL 8: SUCCESS


We now have the complete six-city transition history from 1975→1980 through 2015→2020.

Main historical finding

The proportion of cells classified as stable_rural consistently declined:

Transition	Stable-rural cells	Share
1975→1980	24,089	81.94%
1985→1990	22,550	76.70%
1995→2000	21,186	72.06%
2005→2010	19,377	65.91%
2010→2015	18,509	62.96%
2015→2020	17,108	58.19%

Across the six metropolitan study regions, the stable-rural share fell by 23.75 percentage points between the first and last transition periods.

At the same time, stable_urban_centre increased:

Transition	Stable urban-centre share
1975→1980	9.47%
1995→2000	15.00%
2005→2010	18.96%
2015→2020	21.95%

That gives you a clear report narrative: the metropolitan landscapes became progressively more urban, while established urban centres occupied an increasing share of the analytical grid.

Rural-to-urban transitions accelerated sharply in 2015→2020
Transition	Rural-to-urban cells	Share
1975→1980	799	2.72%
1980→1985	789	2.68%
1985→1990	714	2.43%
1990→1995	708	2.41%
1995→2000	653	2.22%
2000→2005	948	3.22%
2005→2010	812	2.76%
2010→2015	793	2.70%
2015→2020	1,355	4.61%

The final period has the largest rural-to-urban conversion:

1,355 cells
19.88 million m² additional built surface
225,018 additional population
Average built-surface increase of 14,673 m² per transitioning cell

Compared with 2010→2015, the number of rural-to-urban cells increased by approximately 71%:

1355−793793×100≈70.9%\frac{1355-793}{793}\times100 \approx 70.9\%7931355−793​×100≈70.9%
Strongest rural-to-urban change by city, 2015→2020
City	Converted cells	Share of city grid	Built-surface growth
Bengaluru	404	8.24%	5.50 million m²
Hyderabad	291	5.94%	3.66 million m²
Pune Metropolitan Region	282	5.76%	5.04 million m²
Delhi NCR	241	4.92%	4.30 million m²
Chennai Metropolitan Region	100	2.04%	1.08 million m²
Mumbai Metropolitan Region	37	0.76%	0.31 million m²

Interpretation
Bengaluru had the broadest spatial rural-to-urban conversion, with 404 cells changing class.
Pune had the largest mean built-surface increase per converted cell, approximately 17,866 m².
Delhi NCR was nearly identical at approximately 17,860 m² per converted cell.
Mumbai had relatively few newly urbanized cells, indicating a more mature and already-urbanized metropolitan grid.
New urban centres

The number of new_urban_centre cells peaked in several earlier periods:

Transition	New urban-centre cells
1975→1980	273
1980→1985	321
1985→1990	404
1990→1995	660
1995→2000	507
2000→2005	660
2005→2010	421
2010→2015	479
2015→2020	355

The last period still produced 355 new urban-centre cells, but the count remained below the peaks seen in 1990→1995 and 2000→2005.

This suggests a shift from the emergence of entirely new centres toward expansion and consolidation of existing urban areas.

Urban regression is limited

urban_regression remains below 1% in every period:

Maximum: 0.64% in 1990→1995
Latest: 0.13% in 2015→2020

Population change in this category is consistently negative, which supports the semantic interpretation of regression rather than random label movement.

One important classification nuance

The categories are mutually exclusive because the notebook assigns them in priority order:

new_urban_centre
rural_to_urban
stable_urban_centre
urban_intensification
urban_regression
stable_rural
stable_other

Therefore, a rural cell becoming an urban centre is counted as new_urban_centre, not rural_to_urban. For a broader measure of urban conversion, combine:

Plain Text
rural_to_urban + new_urban_centre

For 2015→2020:

1,355+355=1,7101,355 + 355 = 1,7101,355+355=1,710

That means 1,710 cells, or approximately 5.82% of the entire six-city grid, underwent a newly urbanized or new-centre transition.

In [12]:
# ============================================================
# CELL 9: CREATE REPORT FIGURES
# ============================================================
import matplotlib.pyplot as plt
FIG=ARTIFACT_ROOT/'figures';FIG.mkdir(exist_ok=True)
pivot=period_df.pivot_table(index='to_epoch',columns='transition_category',values='cell_count',fill_value=0)
ax=pivot.plot(kind='area',stacked=True,figsize=(12,6),title='Six-city settlement transition categories by target epoch')
ax.set_xlabel('Target epoch');ax.set_ylabel('Grid-cell transitions');plt.tight_layout();plt.savefig(FIG/'transition_categories_by_epoch.png',dpi=180);plt.close()
rural=category_df[category_df.transition_category=='rural_to_urban'].pivot_table(index='to_epoch',columns='city_slug',values='cell_count',fill_value=0)
ax=rural.plot(kind='line',marker='o',figsize=(12,6),title='Rural-to-urban transitions by city and target epoch')
ax.set_xlabel('Target epoch');ax.set_ylabel('Grid cells');plt.tight_layout();plt.savefig(FIG/'rural_to_urban_by_city.png',dpi=180);plt.close()
centres=category_df[category_df.transition_category=='new_urban_centre'].pivot_table(index='to_epoch',columns='city_slug',values='cell_count',fill_value=0)
ax=centres.plot(kind='bar',figsize=(12,6),title='New urban-centre transitions by city and target epoch')
ax.set_xlabel('Target epoch');ax.set_ylabel('Grid cells');plt.tight_layout();plt.savefig(FIG/'new_urban_centres_by_city.png',dpi=180);plt.close()
print('Figures created:',len(list(FIG.glob('*.png'))))
print('CELL 9: SUCCESS')

Figures created: 3
CELL 9: SUCCESS


In [13]:
# ============================================================
# CELL 10: FINAL MANIFEST AND COMPLETION GATE
# ============================================================
manifest={
 'created_at_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
 'purpose':'Six-city settlement-transition matrices across all nine five-year transitions',
 'cities':INVENTORY['training_city_slugs'],
 'epochs':EXPECTED_EPOCHS,
 'cell_transition_rows':BUILD_SUMMARY['cell_transition_rows'],
 'city_periods':BUILD_SUMMARY['city_periods'],
 'categories':BUILD_SUMMARY['categories'],
 'hdfs_outputs':{'cell_transitions':CELL_TRANSITIONS,'smod_class_matrix':SMOD_MATRIX,'settlement_group_matrix':GROUP_MATRIX,'transition_category_summary':CATEGORY_SUMMARY,'six_city_period_summary':PERIOD_SUMMARY},
 'drive_outputs':str(ARTIFACT_ROOT),
 'quality':QUALITY,
 'source_datasets_modified':False,
 'india_wide_complete':False,
}
path=ARTIFACT_ROOT/'six_city_settlement_transition_manifest.json';path.write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n')
check_cmd="source /home/ubuntu/ghsl-sedona-env.sh; export HADOOP_HOME=/usr/local/hadoop; export PATH=$HADOOP_HOME/bin:$PATH; for p in '"+CELL_TRANSITIONS+"' '"+SMOD_MATRIX+"' '"+GROUP_MATRIX+"' '"+CATEGORY_SUMMARY+"' '"+PERIOD_SUMMARY+"'; do hdfs dfs -test -e \"$p\" || exit 1; done; echo SETTLEMENT_OUTPUTS_EXIST"
r=run_remote(check_cmd,300)
print('Six cities covered: 6/6')
print('Five-year transitions covered: 9/9')
print('Cell-transition records:',BUILD_SUMMARY['cell_transition_rows'])
print('SMOD transition matrix: COMPLETE')
print('Settlement-group transition matrix: COMPLETE')
print('Proposal-facing transition categories: COMPLETE')
print('Six-city period comparison: COMPLETE')
print('CSV exports: COMPLETE')
print('Report figures: COMPLETE')
print('Source HDFS datasets modified: NO')
print('India-wide transition matrices: NOT YET COMPLETE')
print('NOTEBOOK_14_SIX_CITY_SETTLEMENT_TRANSITIONS_SUCCESS')
print('CELL 10: SUCCESS')

SETTLEMENT_OUTPUTS_EXIST

REMOTE STDERR:

Six cities covered: 6/6
Five-year transitions covered: 9/9
Cell-transition records: 264600
SMOD transition matrix: COMPLETE
Settlement-group transition matrix: COMPLETE
Proposal-facing transition categories: COMPLETE
Six-city period comparison: COMPLETE
CSV exports: COMPLETE
Report figures: COMPLETE
Source HDFS datasets modified: NO
India-wide transition matrices: NOT YET COMPLETE
NOTEBOOK_14_SIX_CITY_SETTLEMENT_TRANSITIONS_SUCCESS
CELL 10: SUCCESS


Outputs:
/content/drive/MyDrive/GHSL_Urban_Growth_Model/
six_city_settlement_transitions_v1/

This folder contains:

smod_class_matrix.csv
settlement_group_matrix.csv
transition_category_summary.csv
six_city_period_summary.csv
export_manifest.json
six_city_settlement_transition_manifest.json

Figures are stored under:


/content/drive/MyDrive/GHSL_Urban_Growth_Model/
six_city_settlement_transitions_v1/figures/

Expected figures:


transition_categories_by_epoch.png
rural_to_urban_by_city.png
new_urban_centres_by_city.png
HDFS: complete analytical datasets

/data/ghsl/analytics/six_city_settlement_transitions_v1/cell_transitions
/data/ghsl/analytics/six_city_settlement_transitions_v1/smod_class_matrix
/data/ghsl/analytics/six_city_settlement_transitions_v1/settlement_group_matrix
/data/ghsl/analytics/six_city_settlement_transitions_v1/transition_category_summary
/data/ghsl/analytics/six_city_settlement_transitions_v1/six_city_period_summary

Strong findings already available

Stable-rural share declined from 81.94% in 1975→1980 to 58.19% in 2015→2020.

Stable urban-centre share increased from 9.47% to 21.95%.

Rural-to-urban transitions reached their highest level in 2015→2020:
1,355 cells

4.61% of the six-city grid

19.88 million m² additional built surface

Including new urban-centre transitions, 1,710 cells, or 5.82%, became newly urban or new urban centres in 2015→2020.

Bengaluru had the broadest 2015→2020 rural-to-urban conversion:
404 cells

8.24% of its analytical grid

Pune and Delhi had the largest mean built-surface growth per converted cell:

Pune: approximately 17,866 m²

Delhi NCR: approximately 17,860 m²
